# Quando a chuva vira problema

**Chuva e ocorrências urbanas nos bairros do Rio de Janeiro**

Pedro Henrique Vasconcelos Neves e Gabriel Andrade Souza · Análise Exploratória de Dados e Visualização · FGV EMAp 2026.2

A ideia: quanta chuva, e em que lugar, vira problema no Rio? E por que a mesma chuva causa estragos diferentes em bairros diferentes?

## 1. De onde vêm os dados

| Base | Fonte | O que tem |
|---|---|---|
| Chuva | Pluviômetros do [Sistema Alerta Rio](https://www.sistema-alerta-rio.com.br/download/dados-meteorologicos/) | Acumulados de 15 min a 96 h, a cada 15 min, desde 1997 |
| Ocorrências | [Centro de Operações Rio (COR)](https://cor.rio/historia/), [publicadas no datalake](https://cor.rio/prefeitura-do-rio-lanca-dados-do-cor-em-datalake-disponivel-ao-cidadao/) | Cada evento registrado: tipo, gravidade, local, início e fim, órgãos acionados |
| Bairros | Limites de bairro do IPP (datalake da prefeitura) | Polígono de cada bairro |
| Censo 2022 | [Censo 2022 por bairro](https://www.data.rio/datasets/fd354740f1934bf5bf8e9b0e2b509aa9_2/about) (IBGE, agregado pelo IPP) | População e domicílios |
| IPS 2022 | [IPS Bairros 2022](https://ips-rio-pcrj.hub.arcgis.com/pages/ipsbairros) | Índice de Progresso Social e indicadores como população em favelas não urbanizadas e domicílios inadequados |

Chuva, ocorrências e bairros estão no **datalake público da prefeitura** (BigQuery, projeto `datario`). Censo e IPS estão no **Data.Rio**.

## 2. Baixando

O datalake é público: basta uma conta Google e uma consulta SQL. Por exemplo:

```python
pandas_gbq.read_gbq("SELECT * FROM `datario.adm_cor_comando.ocorrencias`", project_id="<seu projeto>")
```

Para não repetir o download, guardamos tudo em `dados/brutos/` com dois scripts (cerca de 1,5 GB):

In [1]:
# Já rodado; descomentar para baixar de novo
# !python scripts/baixar_datario.py
# !python scripts/baixar_data_rio.py

## 3. Uma olhada nos dados

In [2]:
from datetime import date
from pathlib import Path

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", None)

BQ = Path("dados/brutos/bigquery")
DATA_RIO = Path("dados/brutos/data_rio")

### Cobertura e qualidade dos dados

Antes de explorar as relações entre chuva e ocorrências, conferimos a cobertura das bases completas e a ausência de campos essenciais. Nas ocorrências, usamos a data de início no horário do Rio; na chuva, a data registrada em `data_particao`. O recorte comum é a interseção desses períodos, sem presumir que todos os dias e estações tenham medições completas.

Os percentuais abaixo contam registros com data ausente (data ou horário, na chuva) e com pelo menos uma coordenada ausente. As coordenadas dos pluviômetros ficam no cadastro de estações, não nas leituras. Esta etapa apenas descreve os dados, sem excluir registros.


In [3]:
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Colunas da base original, antes de qualquer junção ou filtro.
cor_cobertura = pd.read_parquet(
    BQ / "adm_cor_comando" / "ocorrencias.parquet",
    columns=["data_inicio", "latitude", "longitude"],
)
estacoes_cobertura = pd.read_parquet(
    BQ / "clima_pluviometro" / "estacoes_alertario.parquet",
    columns=["id_estacao", "latitude", "longitude"],
)
datas_cor = cor_cobertura["data_inicio"].dt.tz_convert("America/Sao_Paulo").dt.date
coords_cor_ausentes = cor_cobertura[["latitude", "longitude"]].isna().any(axis=1)
coords_estacoes_ausentes = estacoes_cobertura[["latitude", "longitude"]].isna().any(axis=1)

# Referência fixa para tornar a identificação de datas futuras reproduzível.
data_referencia = date(2026, 10, 28)
datas_chuva_futuras = 0

# A série inteira de chuva é lida em lotes, usando apenas três colunas.
arquivo_chuva = pq.ParquetFile(BQ / "clima_pluviometro" / "taxa_precipitacao_alertario.parquet")
inicio_chuva, fim_chuva = None, None
n_chuva, datas_chuva_ausentes = 0, 0
ids_estacoes_chuva = set()
for lote in arquivo_chuva.iter_batches(columns=["data_particao", "horario", "id_estacao"]):
    n_chuva += lote.num_rows
    limites = pc.min_max(lote["data_particao"]).as_py()
    if limites["min"] is not None:
        inicio_chuva = min(inicio_chuva, limites["min"]) if inicio_chuva else limites["min"]
        fim_chuva = max(fim_chuva, limites["max"]) if fim_chuva else limites["max"]
    datas_chuva_ausentes += pc.sum(
        pc.or_(pc.is_null(lote["data_particao"]), pc.is_null(lote["horario"]))
    ).as_py()
    datas_chuva_futuras += pc.sum(
        pc.fill_null(pc.greater(lote["data_particao"], data_referencia), False)
    ).as_py()
    ids_estacoes_chuva.update(pc.unique(lote["id_estacao"]).to_pylist())
ids_estacoes_chuva.discard(None)

inicio_cor, fim_cor = datas_cor.dropna().min(), datas_cor.dropna().max()
cobertura = pd.DataFrame([
    {"Base": "Ocorrências COR", "Registros": len(cor_cobertura),
     "Início": inicio_cor, "Fim": fim_cor, "Estações": "Não se aplica",
     "Datas ausentes (%)": datas_cor.isna().mean() * 100,
     "Datas futuras (n)": int((datas_cor.dropna() > data_referencia).sum()),
     "Coordenadas ausentes (%)": round(coords_cor_ausentes.mean() * 100, 2)},
    {"Base": "Chuva Alerta Rio", "Registros": n_chuva,
     "Início": inicio_chuva, "Fim": fim_chuva, "Estações": len(ids_estacoes_chuva),
     "Datas ausentes (%)": 100 * datas_chuva_ausentes / n_chuva,
     "Datas futuras (n)": datas_chuva_futuras,
     "Coordenadas ausentes (%)": "Ver cadastro"},
])
display(cobertura.round(2))
print(f"Cadastro: {len(estacoes_cobertura)} estações; "
      f"{coords_estacoes_ausentes.mean():.2%} com coordenadas ausentes.")

inicio_comum, fim_comum = max(inicio_cor, inicio_chuva), min(fim_cor, fim_chuva)
if inicio_comum <= fim_comum:
    print(f"Recorte comum adotado para os próximos cruzamentos: {inicio_comum:%d/%m/%Y} a {fim_comum:%d/%m/%Y}.")
else:
    print("As bases não têm sobreposição temporal; é necessário rever o recorte.")


,Base,Registros,Início,Fim,Estações,Datas ausentes (%),Datas futuras (n),Coordenadas ausentes (%)
0,Ocorrências COR,91406,2015-02-23,2024-04-02,Não se aplica,0.00,0,0.2
1,Chuva Alerta Rio,29345149,1997-01-01,2099-03-03,33,0.02,1,Ver cadastro


Cadastro: 37 estações; 10.81% com coordenadas ausentes.
Recorte comum adotado para os próximos cruzamentos: 23/02/2015 a 02/04/2024.


**Como interpretar:** início e fim indicam a extensão observada dos arquivos, não cobertura contínua. Zero de chuva é uma medição válida e não deve ser confundido com ausência de leitura. O percentual de coordenadas ausentes também não verifica se os pontos estão dentro do município. Antes dos cruzamentos, ainda será necessário avaliar lacunas por estação, coordenadas inválidas e compatibilidade dos horários entre as fontes. O recorte comum substitui a previsão de 2015–2025 da proposta; a série mais longa de chuva pode ser usada separadamente na análise climática.

**Alerta encontrado:** há datas de chuva chegando a 2099, posteriores à referência de 28/10/2026. Elas foram contadas como datas futuras. A subseção seguinte documenta a inspeção e o tratamento para as análises, preservando a base bruta.


### Tratamento da data futura na chuva

Inspecionamos o registro posterior à referência de 28/10/2026 e as últimas leituras com datas não futuras da mesma estação. A inspeção usa a base bruta e não altera o arquivo.


In [ ]:
import pyarrow.dataset as ds

base_chuva = ds.dataset(
    BQ / "clima_pluviometro" / "taxa_precipitacao_alertario.parquet",
    format="parquet",
)
filtro_futuro = ds.field("data_particao") > data_referencia
registros_futuros = base_chuva.to_table(filter=filtro_futuro).to_pandas()
display(registros_futuros)

# Contexto: últimas três leituras distintas, com data não futura, da estação afetada.
ids_afetados = registros_futuros["id_estacao"].dropna().unique().tolist()
ultimas_leituras = base_chuva.to_table(
    columns=["primary_key", "data_particao", "horario", "acumulado_chuva_24_h"],
    filter=(ds.field("id_estacao").isin(ids_afetados))
    & (ds.field("data_particao") <= data_referencia),
).to_pandas().drop_duplicates().sort_values(["data_particao", "horario"]).tail(3)
display(ultimas_leituras)

# Visão para as análises: exclui apenas datas futuras; ausências ficam para outra etapa.
# O filtro é aplicado na leitura e não regrava o Parquet bruto.
chuva_para_analise = base_chuva.filter(
    (ds.field("data_particao") <= data_referencia)
    | ds.field("data_particao").is_null()
)
print(f"Registros com data futura excluídos da análise: {len(registros_futuros)}.")
print(f"Registros restantes: {chuva_para_analise.count_rows():,}.")


primary_key,id_estacao,acumulado_chuva_15_min,acumulado_chuva_1_h,acumulado_chuva_4_h,acumulado_chuva_24_h,acumulado_chuva_96_h,horario,data_particao
28_2099-03-03 03:30:00,28,NaN,NaN,NaN,NaN,NaN,03:30:00,2099-03-03


primary_key,data_particao,horario,acumulado_chuva_24_h
28_2024-06-02 23:45:00,2024-06-02,23:45:00,0.0
28_2024-06-02 23:50:00,2024-06-02,23:50:00,0.0
28_2024-06-03 00:00:00,2024-06-03,00:00:00,0.0


Registros com data futura excluídos da análise: 1.
Registros restantes: 29,345,148.


**Decisão:** o único registro futuro é da estação 28 (Alto da Boa Vista), em 03/03/2099 às 03:30. A própria `primary_key` contém essa data e todos os cinco acumulados de chuva estão ausentes. As últimas leituras não futuras da estação chegam a 03/06/2024; isso não permite deduzir a data correta do registro. Por isso, não imputamos uma data: excluímos datas posteriores à referência de 28/10/2026 da visão `chuva_para_analise`, usada nas leituras seguintes. A tabela de cobertura acima continua descrevendo a base bruta, inclusive a anomalia. Nenhum outro tratamento de qualidade foi aplicado aqui.


### Ocorrências do COR

In [4]:
# Um evento por linha, de 2015 a abril de 2024
ocorrencias = pd.read_parquet(BQ / "adm_cor_comando" / "ocorrencias.parquet")
print(ocorrencias.shape)
ocorrencias.head()

(91406, 13)


,id_pop,id_evento,bairro,data_inicio,data_fim,prazo,descricao,gravidade,latitude,longitude,status,tipo,data_particao
0,1,10707,NaN,2016-10-01 18:58:16+00:00,2016-10-01 19:37:23+00:00,NaN,Autoestrada grajaú - jacarepaguá - km 4 - jaca...,Baixo,-22.916662,-43.262981,Fechado,Primario,2016-10-01
1,1,10693,NaN,2016-10-01 12:31:53+00:00,2016-10-01 13:07:23+00:00,NaN,"R. andré rocha, 1025 b - taquara",Baixo,-22.925347,-43.370468,Fechado,Primario,2016-10-01
2,1,10701,NaN,2016-10-01 17:01:26+00:00,2016-10-01 17:44:41+00:00,NaN,"Estr. das capoeiras, 80 - campo grande",Baixo,-22.895124,-43.558608,Fechado,Primario,2016-10-01
3,1,10708,NaN,2016-10-01 19:32:56+00:00,2016-10-01 20:07:42+00:00,NaN,"Av. das américas, 12600 - barra da tijuca",Baixo,-23.002726,-43.426523,Fechado,Primario,2016-10-01
4,1,10700,NaN,2016-10-01 17:00:14+00:00,2016-10-01 17:45:07+00:00,NaN,"Av. das américas, 3959 - barra",Baixo,-23.001086,-43.352867,Fechado,Primario,2016-10-01


In [5]:
# O tipo de evento vem como código (id_pop); outra tabela dá o nome de cada código
pop = pd.read_parquet(BQ / "adm_cor_comando" / "procedimento_operacional_padrao.parquet")
ocorrencias = ocorrencias.merge(pop, on="id_pop", how="left")
ocorrencias["pop_titulo"].value_counts().head(10)

pop_titulo
Acidente/enguiço sem vítima              41611
Acidente com vítima(s)                   19286
Operação policial                         5060
Bolsão d'água em via                      4231
Queda de árvore                           3695
Obra em local público                     2945
Atropelamento                             2430
Evento em local público ou particular     2277
Manifestação em local público             2192
Incêndio em veículo(s)                    1300
Name: count, dtype: int64

### Chuva

In [6]:
# Os pluviômetros do Alerta Rio: onde ficam e se estão operando
estacoes = pd.read_parquet(BQ / "clima_pluviometro" / "estacoes_alertario.parquet")
print(estacoes.shape)
estacoes.head()

(37, 12)


,id_estacao,estacao,latitude,longitude,cota,x,y,endereco,situacao,data_inicio_operacao,data_fim_operacao,data_atualizacao
0,1,Vidigal,-22.99250,-43.23306,85.0,681138532.0,7.456241e+09,"Hotel Sheraton - Av. Niemeyer,121",Operante,NaT,NaT,NaT
1,2,Urca,-22.95583,-43.16667,90.0,688004213.0,7.460236e+09,"Instituto Militar de Engenharia - Av. Pasteur,35",Operante,NaT,NaT,NaT
2,3,Rocinha,-22.98583,-43.24500,160.0,679831802.0,7.457041e+09,"Região Administrativa - Estrada da Gávea,242",Operante,NaT,NaT,NaT
3,4,Tijuca,-22.93194,-43.22167,340.0,682358108.0,7.462941e+09,Centro de Estudos do Sumaré - Estrada do Sumar...,Operante,NaT,NaT,NaT
4,5,Santa teresa,-22.93167,-43.19639,170.0,684951792.0,7.462972e+09,Escola Estadual Monteiro de Carvalho - Rua Alm...,Operante,NaT,NaT,NaT


In [7]:
# Lemos apenas o temporal de 8 de abril de 2019, usando a visão sem datas futuras.
chuva = chuva_para_analise.to_table(
    filter=ds.field("data_particao") == date(2019, 4, 8),
).to_pandas()
print(chuva.shape)
chuva.head()

(3168, 9)


,primary_key,id_estacao,acumulado_chuva_15_min,acumulado_chuva_1_h,acumulado_chuva_4_h,acumulado_chuva_24_h,acumulado_chuva_96_h,horario,data_particao
0,1_2019-04-08 02:15:00,1,0.0,0.0,0.0,0.0,0.0,02:15:00,2019-04-08
1,1_2019-04-08 19:45:00,1,10.8,44.4,71.6,71.6,71.6,19:45:00,2019-04-08
2,1_2019-04-08 10:00:00,1,0.0,0.0,0.0,0.0,0.0,10:00:00,2019-04-08
3,1_2019-04-08 18:00:00,1,2.4,20.6,20.6,20.6,20.6,18:00:00,2019-04-08
4,10_2019-04-08 03:15:00,10,0.0,0.0,0.0,0.0,0.0,03:15:00,2019-04-08


In [8]:
# Maior acumulado em 24 h em cada estação nesse dia (mm)
(
    chuva.groupby("id_estacao")["acumulado_chuva_24_h"].max()
    .rename("max_24h_mm")
    .reset_index()
    .merge(estacoes[["id_estacao", "estacao"]], on="id_estacao")
    .sort_values("max_24h_mm", ascending=False)
    .head(10)
)

,id_estacao,max_24h_mm,estacao
8,17,225.0,Barrinha
7,16,219.8,Jardim botanico
29,6,218.0,Copacabana
22,3,208.2,Rocinha
20,28,208.0,Alto da boa vista
10,19,184.6,Riocentro
0,1,183.2,Vidigal
11,2,165.8,Urca
9,18,157.8,Cidade de deus
14,22,136.0,Santa cruz


### O que a cidade registrou nesse temporal

In [9]:
# Ocorrências abertas em 8 e 9 de abril de 2019, por tipo
temporal = ocorrencias[ocorrencias["data_inicio"].dt.date.isin([date(2019, 4, 8), date(2019, 4, 9)])]
print(len(temporal), "ocorrências")
temporal["pop_titulo"].value_counts().head(10)

159 ocorrências


pop_titulo
Bolsão d'água em via                    58
Queda de árvore                         41
Acidente/enguiço sem vítima             21
Alagamento                              13
Deslizamento                             5
Afundamento de pista / buraco na via     3
Acidente com vítima(s)                   3
Lâmina d'água                            3
Alagamentos e enchentes                  2
Atropelamento                            2
Name: count, dtype: int64

### Bairros e perfil

In [10]:
# Limites dos bairros: a unidade de comparação
bairros = gpd.read_parquet(BQ / "dados_mestres" / "bairro.parquet")
print(bairros.shape)
bairros.head()

(167, 12)


,id_bairro,nome,id_area_planejamento,id_regiao_planejamento,nome_regiao_planejamento,id_regiao_administrativa,nome_regiao_administrativa,subprefeitura,area,perimetro,geometry_wkt,geometry
0,3,Santo Cristo,1,1.1,Centro,1,Portuaria,Centro,1.684725e+06,6743.227885,POLYGON ((-43.194498082949565 -22.903378004069...,"POLYGON ((-43.19459 -22.90326, -43.19474 -22.9..."
1,1,Saúde,1,1.1,Centro,1,Portuaria,Centro,3.638186e+05,2646.220568,POLYGON ((-43.1811516335027 -22.89543028498111...,"POLYGON ((-43.18119 -22.89542, -43.18132 -22.8..."
2,4,Caju,1,1.1,Centro,1,Portuaria,Centro,5.347495e+06,19800.522524,POLYGON ((-43.22522241788446 -22.8746498473696...,"MULTIPOLYGON (((-43.22522 -22.87464, -43.22521..."
3,2,Gamboa,1,1.1,Centro,1,Portuaria,Centro,1.112906e+06,4612.833630,POLYGON ((-43.18791509600112 -22.8931217219090...,"POLYGON ((-43.19069 -22.89216, -43.19093 -22.8..."
4,161,Lapa,1,1.1,Centro,2,Centro,Centro,2.983266e+05,3849.181818,POLYGON ((-43.18166120770175 -22.9120798231449...,"POLYGON ((-43.18217 -22.91194, -43.18243 -22.9..."


In [11]:
# Censo 2022: população e domicílios por bairro
censo = gpd.read_parquet(DATA_RIO / "censo2022_bairros.parquet")
print(censo.shape)
censo.head()

(165, 18)


,nome,codbairro,regiao_adm,codra,area_plane,Total_de_pessoas_2010,Total_de_pessoas_2022,diferenca_de_pessoas,Total_de_domicilios_2022,Total_DP_2022,Total_DC,Total_DP_Ocup_2010,Total_DP_Ocup_2022,diferenca_de_DP,Shape_Length,Shape_Area,OBJECTID_1,geometry
0,Grumari,133,BARRA DA TIJUCA,24,4,167,184,17,85,85,0,44,68,24,23375.374038,9.598851e+06,1,"MULTIPOLYGON (((-43.50819 -23.03311, -43.50882..."
1,Jardim Sulacap,137,REALENGO,33,5,13062,14396,1334,6389,6387,2,4390,5481,1091,14519.328424,7.869225e+06,2,"POLYGON ((-43.37541 -22.88962, -43.37541 -22.8..."
2,Saúde,001,PORTUARIA,1,1,2749,1876,-873,1001,988,13,946,801,-145,2646.220568,3.638186e+05,3,"POLYGON ((-43.18115 -22.89543, -43.18119 -22.8..."
3,Magalhães Bastos,138,REALENGO,33,5,24430,20938,-3492,9340,9336,4,8092,8045,-47,6902.663944,1.975955e+06,4,"POLYGON ((-43.40929 -22.88269, -43.40923 -22.8..."
4,Realengo,139,REALENGO,33,5,180123,165881,-14242,72817,72801,16,59259,63429,4170,30896.866166,2.605427e+07,5,"POLYGON ((-43.43517 -22.8419, -43.43516 -22.84..."


In [12]:
# IPS 2022: nota geral e por componente, por bairro
ips = gpd.read_parquet(DATA_RIO / "ips2022_bairros.parquet")
print(ips.shape)
ips.head()

(159, 51)


,objectid,bairro,bairrostr,ra,rastr,subp,rp,rpstr,ap,apstr,ncm_2022,as_2022,m_2022,sp_2022,acb_2022,aic_2022,sbe_2022,qma_2022,di_2022,lie_2022,ti_2022,aes_2022,nhb_2022,fbe_2022,opo_2022,ips_2022,ranking_2022,cluster_2022,ipsra_med_2022,ranking_ra_2022,renda_2010,nhb_2022_mapa,fbe_2022_mapa,opo_2022_mapa,ips_2022_mapa,GlobalID,Shape__Area,Shape__Length,ncm_2022_mapa,as_2022_mapa,m_2022_mapa,sp_2022_mapa,acb_2022_mapa,aic_2022_mapa,sbe_2022_mapa,qma_2022_mapa,di_2022_mapa,lie_2022_mapa,ti_2022_mapa,aes_2022_mapa,geometry
0,1,1,Saúde,1.0,I - R.A. Portuária,Centro e Centro Histórico,11.0,1.1 - Centro,1.0,AP1,41.063350,88.112062,77.456069,89.901783,18.809620,46.224159,95.364943,54.034396,46.922688,79.403994,56.292418,4.707666,74.133316,53.608279,46.831692,58.191096,137,4,57.138098,29,759.8613,41.21 - 81.38,52.97 - 57.4,35.6 - 48.37,50.43 - 60.59,d8864e2b-aca4-4d05-9382-74d5321fd553,3.357000e+05,2900.590638,31.07 - 55.96,24.7 - 90.03,53.34 - 85.37,12.24 - 90.27,10.97 - 24.99,16.01 - 63.76,86.8 - 95.44,54.01 - 70.77,43.59 - 48.49,78.96 - 83.52,31.22 - 65.87,2.38 - 11.34,"POLYGON ((-43.1864 -22.89361, -43.18128 -22.89..."
1,2,2,Gamboa,1.0,I - R.A. Portuária,Centro e Centro Histórico,11.0,1.1 - Centro,1.0,AP1,43.138014,85.209814,54.995898,97.936843,16.291654,46.224159,82.414870,35.779938,39.839666,93.355489,66.628457,4.707666,70.320142,45.177655,51.132820,55.543539,146,4,57.138098,29,567.8586,41.21 - 81.38,38.28 - 52.97,48.37 - 53.49,50.43 - 60.59,7c088456-443d-4084-87c2-25cbc03ae91e,1.135798e+06,5774.859101,31.07 - 55.96,24.7 - 90.03,53.34 - 85.37,96.47 - 99.91,10.97 - 24.99,16.01 - 63.76,81.68 - 86.8,8.57 - 39.37,37.74 - 43.59,83.52 - 99.13,65.87 - 70.67,2.38 - 11.34,"POLYGON ((-43.18647 -22.89453, -43.18666 -22.8..."
2,3,3,Santo Cristo,1.0,I - R.A. Portuária,Centro e Centro Histórico,11.0,1.1 - Centro,1.0,AP1,46.578798,89.444286,68.645268,90.101633,27.087956,46.224159,42.763759,39.367534,35.868030,74.430838,68.960598,4.707666,73.692496,38.860852,45.991783,52.848377,153,4,57.138098,29,633.8508,41.21 - 81.38,38.28 - 52.97,35.6 - 48.37,50.43 - 60.59,9ed3acb6-966f-47b2-8b32-c59a008f11a6,1.612239e+06,7314.959711,31.07 - 55.96,24.7 - 90.03,53.34 - 85.37,12.24 - 90.27,24.99 - 29.54,16.01 - 63.76,38.56 - 79.64,39.37 - 46.3,19.6 - 37.74,45.45 - 74.7,65.87 - 70.67,2.38 - 11.34,"POLYGON ((-43.19916 -22.89713, -43.19901 -22.8..."
3,4,4,Caju,1.0,I - R.A. Portuária,Centro e Centro Histórico,11.0,1.1 - Centro,1.0,AP1,70.807500,90.917722,85.291923,91.884834,32.762381,46.224159,77.326878,58.560838,38.326463,76.511428,70.310763,4.707666,84.725495,53.718564,47.464080,61.969380,106,3,57.138098,29,469.6093,82.95 - 87.22,52.97 - 57.4,35.6 - 48.37,60.59 - 64.62,da9338ce-1aee-4645-ac5e-4a9321130d8f,5.016139e+06,20049.396390,69.34 - 82.07,90.48 - 97.39,53.34 - 85.37,91.11 - 96.47,29.54 - 34.48,16.01 - 63.76,38.56 - 79.64,54.01 - 70.77,37.74 - 43.59,74.7 - 78.96,65.87 - 70.67,2.38 - 11.34,"MULTIPOLYGON (((-43.20767 -22.86885, -43.20734..."
4,6,5,Centro,2.0,II - R.A. Centro,Centro e Centro Histórico,11.0,1.1 - Centro,1.0,AP1,52.791429,91.099535,67.881105,49.426361,38.770919,80.452172,82.533756,57.865332,50.364412,80.692383,48.394022,25.543224,65.299608,64.905545,51.248510,60.484554,120,4,60.484554,21,1613.2935,41.21 - 81.38,61.79 - 76.2,48.37 - 53.49,50.43 - 60.59,7e7c3a8e-07ac-4a44-a80a-9edf8a1d9c9d,5.886108e+06,24437.276146,31.07 - 55.96,90.48 - 97.39,53.34 - 85.37,12.24 - 90.27,34.48 - 50.13,72.09 - 83.85,81.68 - 86.8,54.01 - 70.77,48.49 - 80.58,78.96 - 83.52,31.22 - 65.87,25.34 - 50.43,"MULTIPOLYGON (((-43.17939 -22.89528, -43.1795 ..."


## 4. O que vamos tentar ver

1. **Onde e quando chove**: como a chuva se distribui pela cidade, ao longo do ano e do dia? Os dias extremos ficaram mais frequentes desde 1997?
2. **O que a chuva provoca**: que tipos de ocorrência aumentam em dias de chuva, e onde?
3. **O limiar**: a partir de quantos milímetros a chance de ocorrência passa de 50%? A prefeitura fala em 30 mm/dia.
4. **Pancada ou acumulado**: o que explica melhor cada tipo de ocorrência, chuva curta e forte (15 min, 1 h) ou acumulada (24 h, 96 h)?
5. **Bairro a bairro**: o limiar é igual na cidade toda?
6. **Quem é mais vulnerável**: os bairros que mais sofrem com a mesma chuva têm mais favelas não urbanizadas, mais domicílios inadequados, maior densidade?
7. **A resposta**: com a mesma chuva, o tempo de resposta dos órgãos muda entre regiões?

**Como juntar**: cada ocorrência tem coordenada e horário. Ligamos cada uma ao pluviômetro mais próximo e à chuva das horas anteriores, e depois ao bairro onde caiu.

**Cuidados**
- As bases do datalake vão até 2024 (ocorrências até abril).
- Ocorrência registrada não é o mesmo que alagamento real, e áreas mais monitoradas tendem a registrar mais.
- O campo de bairro das ocorrências está vazio em ~40% dos casos, então o bairro vai sair das coordenadas.